# Phase 6: Econometric Analysis & Model Assumption Audit

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Test heteroskedasticity (Breusch-Pagan, White test) and calculate HC3 robust standard errors.
2. Evaluate autocorrelation (Durbin-Watson, Breusch-Godfrey).
3. Assess linearity via Box-Tidwell transformations.
4. Audit outlier influence metrics (Cook's Distance, Hat values).
5. Evaluate 2SLS Instrumental Variable endogeneity framework.

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
from validation.econometrics import run_heteroskedasticity_tests
num_feats = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low"]
het_summary, hc3_table = run_heteroskedasticity_tests(df, "target", num_feats)
print("Heteroskedasticity Test Summary:", het_summary)
hc3_table

Heteroskedasticity Test Summary: {'breusch_pagan_lm_stat': 1244.0614, 'breusch_pagan_pvalue': 1.3914913286510065e-265, 'white_lm_stat': nan, 'white_pvalue': nan, 'goldfeld_quandt_stat': 0.976, 'goldfeld_quandt_pvalue': 0.8538240908996981, 'heteroskedasticity_present': 'Yes'}


,feature,coef,ols_std_err,hc3_robust_std_err,se_difference_pct,ols_pvalue,hc3_pvalue
0,const,0.196598,0.082482,0.076317,-7.48,1.716098e-02,9.992750e-03
1,loan_amnt,0.000017,0.000001,0.000001,16.37,4.825581e-41,6.270815e-31
2,int_rate,0.024463,0.000847,0.000925,9.18,1.942373e-178,4.321010e-154
3,installment,-0.000507,0.000042,0.000048,12.93,4.291078e-33,1.984122e-26
4,annual_inc,-0.000000,0.000000,0.000000,154.04,8.779990e-02,5.015713e-01
5,dti,0.003354,0.000371,0.000401,8.14,1.782517e-19,6.369429e-17
6,fico_range_low,-0.000530,0.000113,0.000102,-9.98,2.742207e-06,1.879427e-07


In [4]:
from validation.diagnostics import run_box_tidwell_test
bt_table = run_box_tidwell_test(df, "target", num_feats)
bt_table

,feature,interaction_coef,box_tidwell_pvalue,is_non_linear
0,loan_amnt,-0.000045,1.120467e-06,Yes
1,int_rate,-0.225532,5.907968e-14,Yes
2,installment,-0.002209,1.461546e-12,Yes
3,annual_inc,0.000001,4.273250e-03,Yes
4,fico_range_low,-0.020322,5.446359e-01,No
